In [ ]:
import pandas as pd
from openpyxl import load_workbook
from openpyxl.formatting.rule import CellIsRule, FormulaRule
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.utils import get_column_letter

market_data_dirty = pd.read_excel(r"market_data_dirty.xlsx")

## Completeness rationale

In a simple setting, completeness could be measured as the ratio of non-null values over the total number of records. However, this approach can be misleading.

For this reason, completeness is evaluated using a more structured approach, distinguishing between core fields and conditionally expected fields.

*Core fields* (such as close price or currency) *are expected to be present for all records*, and their completeness is therefore calculated against *the full dataset*.

On the other hand, several fields depend on the availability of other information. For example, market structure fields such as bid, ask and spread are only meaningful when a valid price exists, while volume is not relevant for certain asset classes (e.g. FX). In these cases, measuring completeness over the full dataset would artificially penalise the metric and not reflect the real data quality.

## Conditional Completeness

Here a conditional completeness framework is applied.

For each field, a *specific scope is defined*. Completeness is then calculated only within this relevant subset.

This approach ensures that completeness metrics reflect actual data quality issues, rather than structural characteristics of the dataset

In [6]:
from src_functions import completeness

In [ ]:
completeness_heatmap = completeness(market_data_dirty)

,date,close,ccy,source,spread,fx_rate,volume,bid,ask
asset,,,,,,,,,
AAPL,1.0,0.916058,1.0,0.700095,1.0,1.0,0.598606,0.898426,0.901288
EURUSD=X,1.0,0.999088,1.0,1.000000,1.0,1.0,NaN,1.000000,1.000000
GLD,1.0,0.897810,1.0,0.451277,1.0,1.0,0.725610,0.808656,1.000000
SPY,1.0,0.964416,1.0,1.000000,1.0,1.0,0.923368,0.649007,0.781457
TLT,1.0,0.964416,1.0,0.920530,1.0,1.0,1.000000,1.000000,0.696814


In [21]:
# Prepare completeness matrix
cols = ["close", "source", "volume", "bid", "ask", "spread"]

df_excel = completeness_heatmap[cols].copy().astype(float) * 100
df_excel = df_excel.round(1)

file_path = r"completeness_heatmap.xlsx"
df_excel.to_excel(file_path, sheet_name="Completeness", index=True)

wb = load_workbook(file_path)
ws = wb["Completeness"]

header_fill = PatternFill(start_color="D9EAF7", end_color="D9EAF7", fill_type="solid")
red_fill = PatternFill(start_color="FF4D4D", end_color="FF4D4D", fill_type="solid")
amber_fill = PatternFill(start_color="FFA500", end_color="FFA500", fill_type="solid")
green_fill = PatternFill(start_color="4CAF50", end_color="4CAF50", fill_type="solid")
gray_fill = PatternFill(start_color="C0C0C0", end_color="C0C0C0", fill_type="solid")

thin_gray = Side(style="thin", color="D9D9D9")
border = Border(left=thin_gray, right=thin_gray, top=thin_gray, bottom=thin_gray)

header_font = Font(name="Arial", size=11, bold=True, color="000000")
asset_font = Font(name="Arial", size=11, bold=True, color="000000")
value_font = Font(name="Arial", size=11, bold=True, color="FFFFFF")

center_align = Alignment(horizontal="center", vertical="center")
left_align = Alignment(horizontal="left", vertical="center")

# Header formatting
for cell in ws[1]:
    cell.font = header_font
    cell.alignment = center_align
    cell.fill = header_fill
    cell.border = border

# Body formatting
max_row = ws.max_row
max_col = ws.max_column

for row in ws.iter_rows(min_row=2, max_row=max_row, min_col=1, max_col=max_col):
    for cell in row:
        cell.border = border

        # colonna A = asset
        if cell.column == 1:
            cell.font = asset_font
            cell.alignment = left_align
        else:
            cell.font = value_font
            cell.alignment = center_align
            cell.number_format = "0.0"

# Column widths
ws.column_dimensions["A"].width = 14

for col_idx in range(2, max_col + 1):
    col_letter = get_column_letter(col_idx)
    ws.column_dimensions[col_letter].width = 11

# Freeze panes
ws.freeze_panes = "B2"

# Row height
for row_idx in range(1, max_row + 1):
    ws.row_dimensions[row_idx].height = 22

# Conditional formatting
end_col_letter = get_column_letter(max_col)
data_range = f"B2:{end_col_letter}{max_row}"

# blank cells -> gray
ws.conditional_formatting.add(
    data_range,
    FormulaRule(formula=["ISBLANK(B2)"], fill=gray_fill)
)

# thresholds on 0-100 scale
ws.conditional_formatting.add(
    data_range,
    CellIsRule(operator="lessThan", formula=["75"], fill=red_fill)
)

ws.conditional_formatting.add(
    data_range,
    CellIsRule(operator="between", formula=["75", "90"], fill=amber_fill)
)

ws.conditional_formatting.add(
    data_range,
    CellIsRule(operator="greaterThanOrEqual", formula=["90"], fill=green_fill)
)

wb.save(file_path)